# ⚙️ Machine Failure Prediction System
**Capstone Project 01 — Predictive Maintenance**

Dataset: `predictive_maintenance.csv` (10,000 rows)  
Models: Logistic Regression · K-Nearest Neighbors  
Target: `Target` column → 0 = No Failure, 1 = Failure

## ✅ Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first — installs any missing packages
!pip install scikit-learn pandas numpy matplotlib seaborn joblib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score
)

print('✅ All libraries imported successfully!')

## ✅ Step 2 — Upload & Load Dataset

In [ ]:
# Upload your CSV file to Colab
from google.colab import files
uploaded = files.upload()   # <-- click and select predictive_maintenance.csv

In [ ]:
# Load the dataset
df = pd.read_csv('predictive_maintenance.csv')

print('✅ Dataset loaded!')
print(f'   Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns   : {list(df.columns)}')
df.head()

## ✅ Step 3 — Explore the Data (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe().round(2)

In [ ]:
# Class distribution — Target column
counts = df['Target'].value_counts()
labels = ['No Failure', 'Failure']
failure_rate = counts[1] / len(df) * 100

print(f'No Failure : {counts[0]:,} ({100-failure_rate:.1f}%)')
print(f'Failure    : {counts[1]:,} ({failure_rate:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(labels, counts.values, color=['#1D9E75', '#E24B4A'], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (Target)', fontweight='bold', pad=12)
axes[0].set_ylabel('Count')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Failure by machine type
type_failure = df.groupby('Type')['Target'].mean().mul(100).sort_values(ascending=False)
type_failure.plot(kind='bar', ax=axes[1], color=['#E24B4A','#BA7517','#378ADD'],
                  edgecolor='white', rot=0)
axes[1].set_title('Failure Rate by Machine Type (%)', fontweight='bold', pad=12)
axes[1].set_xlabel('Machine Type  (H=High, L=Low, M=Medium)')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Failure type breakdown
print('=== Failure Type Counts ===')
print(df['Failure Type'].value_counts())

ft_counts = df['Failure Type'].value_counts()
plt.figure(figsize=(9, 4))
colors = ['#1D9E75','#E24B4A','#BA7517','#378ADD','#534AB7','#D85A30']
bars = plt.barh(ft_counts.index, ft_counts.values, color=colors[:len(ft_counts)], edgecolor='white')
for bar, val in zip(bars, ft_counts.values):
    plt.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontweight='bold')
plt.title('Failure Type Distribution', fontweight='bold', pad=12)
plt.xlabel('Count')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions — failure vs no failure
numeric_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[df['Target']==0][col], bins=40, alpha=0.65,
                 color='#1D9E75', label='No Failure')
    axes[i].hist(df[df['Target']==1][col], bins=40, alpha=0.65,
                 color='#E24B4A', label='Failure')
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

axes[5].set_visible(False)
plt.suptitle('Feature Distributions: Failure vs No Failure', fontweight='bold', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = numeric_cols + ['Target']
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 10},
            square=True)
plt.title('Feature Correlation Matrix', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## ✅ Step 4 — Data Preprocessing

In [ ]:
# 4.1 Drop columns not needed for ML
df_ml = df.drop(columns=['UDI', 'Product ID', 'Failure Type'])

# 4.2 Encode 'Type' column: H=0, L=1, M=2
le = LabelEncoder()
df_ml['Type'] = le.fit_transform(df_ml['Type'])
print('Type encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# 4.3 Feature engineering — two new columns
df_ml['Temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df_ml['Power']     = df['Torque [Nm]'] * df['Rotational speed [rpm]'] / 9549

print('\nFinal feature set:')
print(df_ml.drop(columns=['Target']).columns.tolist())
df_ml.head()

In [ ]:
# 4.4 Split features and target
X = df_ml.drop(columns=['Target'])
y = df_ml['Target']

# 4.5 Train / Test split  (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples     : {X_test.shape[0]:,}')
print(f'Features         : {X_train.shape[1]}')

# 4.6 Feature scaling — REQUIRED for both LR and KNN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('\n✅ Scaling applied (mean=0, std=1)')

## ✅ Step 5 — Train Models

In [ ]:
# ──────────────────────────────────────────
#  MODEL 1: Logistic Regression
# ──────────────────────────────────────────
print('=' * 50)
print('  MODEL 1 — Logistic Regression')
print('=' * 50)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_s, y_train)

lr_pred  = lr_model.predict(X_test_s)
lr_proba = lr_model.predict_proba(X_test_s)[:, 1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_cv  = cross_val_score(lr_model, X_train_s, y_train, cv=5, scoring='accuracy').mean()
lr_auc = roc_auc_score(y_test, lr_proba)

print(f'\nTest Accuracy : {lr_acc*100:.2f}%')
print(f'CV Accuracy   : {lr_cv*100:.2f}%  (5-fold)')
print(f'ROC-AUC       : {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['No Failure', 'Failure']))

In [ ]:
# ──────────────────────────────────────────
#  MODEL 2: K-Nearest Neighbors
# ──────────────────────────────────────────
print('=' * 50)
print('  MODEL 2 — K-Nearest Neighbors')
print('=' * 50)

# Find optimal K via cross-validation
k_range  = range(1, 21)
k_scores = []
for k in k_range:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    score   = cross_val_score(knn_tmp, X_train_s, y_train, cv=5, scoring='accuracy').mean()
    k_scores.append(score)

best_k = list(k_range)[np.argmax(k_scores)]
print(f'\nOptimal K : {best_k}')

# Plot K vs Accuracy
plt.figure(figsize=(9, 4))
plt.plot(k_range, [s*100 for s in k_scores], 'o-', color='#1D9E75', lw=2, markersize=5)
plt.axvline(best_k, color='#E24B4A', linestyle='--', lw=1.5, label=f'Best K = {best_k}')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('CV Accuracy (%)')
plt.title('KNN — Finding Optimal K', fontweight='bold')
plt.legend()
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Train final KNN with best K
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_s, y_train)

knn_pred  = knn_model.predict(X_test_s)
knn_proba = knn_model.predict_proba(X_test_s)[:, 1]

knn_acc = accuracy_score(y_test, knn_pred)
knn_cv  = cross_val_score(knn_model, X_train_s, y_train, cv=5, scoring='accuracy').mean()
knn_auc = roc_auc_score(y_test, knn_proba)

print(f'\nTest Accuracy : {knn_acc*100:.2f}%')
print(f'CV Accuracy   : {knn_cv*100:.2f}%  (5-fold)')
print(f'ROC-AUC       : {knn_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, knn_pred, target_names=['No Failure', 'Failure']))

## ✅ Step 6 — Evaluate & Compare Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Confusion Matrix: LR ---
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Fail','Fail'], yticklabels=['No Fail','Fail'],
            linewidths=1, linecolor='white', annot_kws={'size':13, 'weight':'bold'})
axes[0].set_title(f'Logistic Regression\nAccuracy: {lr_acc*100:.2f}%', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# --- Confusion Matrix: KNN ---
cm_knn = confusion_matrix(y_test, knn_pred)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Fail','Fail'], yticklabels=['No Fail','Fail'],
            linewidths=1, linecolor='white', annot_kws={'size':13, 'weight':'bold'})
axes[1].set_title(f'KNN  (K={best_k})\nAccuracy: {knn_acc*100:.2f}%', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

# --- ROC Curves ---
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, lr_proba)
fpr_knn, tpr_knn, _ = roc_curve(y_test, knn_proba)

axes[2].plot(fpr_lr,  tpr_lr,  lw=2, color='#378ADD', label=f'Logistic Reg  (AUC = {lr_auc:.3f})')
axes[2].plot(fpr_knn, tpr_knn, lw=2, color='#1D9E75', label=f'KNN K={best_k}  (AUC = {knn_auc:.3f})')
axes[2].plot([0,1],[0,1], 'k--', alpha=0.3, label='Random Classifier')
axes[2].fill_between(fpr_knn, tpr_knn, alpha=0.05, color='#1D9E75')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve Comparison', fontweight='bold')
axes[2].legend(loc='lower right', fontsize=9)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
results = pd.DataFrame({
    'Model'              : ['Logistic Regression', f'KNN  (K={best_k})'],
    'Test Accuracy (%)'  : [round(lr_acc*100, 2),  round(knn_acc*100, 2)],
    'CV Accuracy (%)'    : [round(lr_cv*100, 2),   round(knn_cv*100, 2)],
    'ROC-AUC'            : [round(lr_auc, 4),       round(knn_auc, 4)]
})

print('\n' + '=' * 58)
print('           MODEL COMPARISON SUMMARY')
print('=' * 58)
print(results.to_string(index=False))

best_name = f'KNN (K={best_k})' if knn_acc >= lr_acc else 'Logistic Regression'
print(f'\n🏆 Best Model: {best_name}')

In [ ]:
# Feature importance — LR coefficients
feature_names = X.columns.tolist()
coef = lr_model.coef_[0]
colors = ['#E24B4A' if c > 0 else '#1D9E75' for c in coef]

# Sort by absolute value
sorted_idx = np.argsort(np.abs(coef))[::-1]

plt.figure(figsize=(10, 4))
plt.bar([feature_names[i] for i in sorted_idx],
        [coef[i] for i in sorted_idx],
        color=[colors[i] for i in sorted_idx], edgecolor='white')
plt.axhline(0, color='#888', linewidth=0.8)
plt.title('Logistic Regression — Feature Coefficients\n(red = increases failure risk, green = decreases)', fontweight='bold')
plt.ylabel('Coefficient')
plt.xticks(rotation=20, ha='right')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## ✅ Step 7 — Save Models to Google Drive

In [ ]:
# Mount Google Drive so models persist after session ends
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/MachinePrediction/'
import os
os.makedirs(save_path, exist_ok=True)

joblib.dump(lr_model,     save_path + 'lr_model.pkl')
joblib.dump(knn_model,    save_path + 'knn_model.pkl')
joblib.dump(scaler,       save_path + 'scaler.pkl')
joblib.dump(le,           save_path + 'label_encoder.pkl')
joblib.dump(feature_names, save_path + 'feature_names.pkl')

print('✅ Models saved to Google Drive:')
print(f'   {save_path}lr_model.pkl')
print(f'   {save_path}knn_model.pkl')
print(f'   {save_path}scaler.pkl')
print(f'   {save_path}label_encoder.pkl')

## ✅ Step 8 — Predict on New Sensor Reading

In [ ]:
def predict_failure(machine_type, air_temp, proc_temp, rpm, torque, tool_wear,
                    model=knn_model, use_model='KNN'):
    """
    Predict failure probability for a single machine reading.
    
    Parameters:
    -----------
    machine_type : str   — 'L', 'M', or 'H'
    air_temp     : float — Air temperature in Kelvin
    proc_temp    : float — Process temperature in Kelvin
    rpm          : int   — Rotational speed
    torque       : float — Torque in Nm
    tool_wear    : int   — Tool wear in minutes
    """
    type_enc  = le.transform([machine_type])[0]
    temp_diff = proc_temp - air_temp
    power     = torque * rpm / 9549
    
    features = pd.DataFrame([{
        'Type'                       : type_enc,
        'Air temperature [K]'        : air_temp,
        'Process temperature [K]'    : proc_temp,
        'Rotational speed [rpm]'     : rpm,
        'Torque [Nm]'                : torque,
        'Tool wear [min]'            : tool_wear,
        'Temp_diff'                  : temp_diff,
        'Power'                      : power
    }])[feature_names]
    
    scaled = scaler.transform(features)
    prob   = model.predict_proba(scaled)[0][1]
    pred   = model.predict(scaled)[0]
    
    status = '⚠️  FAILURE LIKELY — Schedule maintenance!' if pred == 1 else '✅  NORMAL — No immediate action needed'
    risk   = 'Critical' if prob > 0.6 else 'High' if prob > 0.4 else 'Moderate' if prob > 0.2 else 'Low'
    
    print(f'Model             : {use_model}')
    print(f'Failure Probability: {prob*100:.1f}%')
    print(f'Risk Level        : {risk}')
    print(f'Prediction        : {status}')
    return prob * 100


# ── Test with real-style values from the dataset ──────────────────
print('─' * 50)
print('EXAMPLE 1: Suspicious high-torque reading')
print('─' * 50)
predict_failure('L', air_temp=298.9, proc_temp=309.0, rpm=1410,
                torque=65.7, tool_wear=191)

print()
print('─' * 50)
print('EXAMPLE 2: Normal reading')
print('─' * 50)
predict_failure('M', air_temp=298.5, proc_temp=308.7, rpm=1520,
                torque=38.0, tool_wear=45)